# 🚀 AIC 2026 - Master Pipeline

Notebook này chứa toàn bộ luồng chạy hoàn chỉnh của hệ thống tìm kiếm video (Ensemble Zero-Shot + Projection Head).

### Bước 0: Kiểm tra hệ thống GPU

In [ ]:
!python 0_check_system.py

### Bước 1: Trích xuất Đặc trưng (Ensemble 3 Mô hình)
Chạy qua toàn bộ dữ liệu ảnh và lưu dưới dạng `.npy`, đồng thời tạo FAISS index tạm thời (2304 chiều).

In [ ]:
!python 1_extract_and_build_index.py

### Bước 2: Huấn luyện Mạng thần kinh (Projection Head)
Dùng các file đặc trưng đã trích xuất ở Bước 1 kết hợp với file `captions_dummy.json` (hoặc nhãn từ BTC) để dạy cho máy học cách khớp câu tiếng Việt vào hình ảnh.

Bạn có thể đổi số `--epochs 50` thành số vòng lặp mà bạn muốn.

In [ ]:
!python 2_train_projection_head.py --epochs 50

### Bước 3: Rebuild FAISS Index
Sau khi mô hình Projection Head `projection_head_latest.pth` được lưu ở Bước 2. Ta sẽ chạy toàn bộ vector ảnh (2304d) qua mô hình này để ép về chuẩn 768d của văn bản, rồi lưu lại thành FAISS Index mới.

In [ ]:
!python 3_rebuild_projected_index.py

### Bước 4: Khởi động Backend API / Hoặc Query Trực Tiếp
Sau khi có FAISS Index hoàn chỉnh, bạn có thể chạy API để Web App Frontend kết nối tới.

In [ ]:
!python main.py

### (Tùy chọn) Chạy Test Truy vấn Trực tiếp trên Notebook

In [ ]:
import sys
import os
from backend.embedding.search_engine import VectorSearchEngine

engine = VectorSearchEngine()
engine.load_index()

query = "một bức ảnh về chiếc xe màu đỏ"
results = engine.search_single(query, top_k=5)

print(f"Kết quả cho truy vấn: {results['query_vi']}")
for r in results['results']:
    print(f"- Video ID: {r['video_id']} | Frame: {r['frame_id']} | Score: {r['score']:.4f}")